# Logistic Regression

This notebook trains and evaluates Logistic Regression using the shared
dataset split, preprocessing pipeline, cross-validation approach and metrics.
Logistic Regression is a simple linear classification algorithm. It
calculates the probability that a booking belongs to the cancelled class,
then uses that probability to classify the booking. It is used as a simple
baseline because it is interpretable and works well with scaled numeric
features and one-hot encoded categories.

In [ ]:
# pyright: reportUnusedImport=false

from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/processed/hotel_booking_features.csv").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from scripts.model_evaluation_utils import (
    RANDOM_STATE,
    OUTPUT_DIR,
    load_feature_dataset,
    split_feature_dataset,
    class_distribution,
    build_pipeline,
    get_cv,
    cross_validation_summary,
    fit_and_evaluate,
    confusion_matrix_frame,
)

FIGURE_DIR = OUTPUT_DIR / "Figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Shared Data Split

This section loads the final feature-engineered dataset and applies the
same train/test split used by the other model notebooks. The class
distribution is shown first so the later precision, recall and F1 results
can be interpreted in relation to the cancellation imbalance.

In [ ]:
data = load_feature_dataset()
prepared = split_feature_dataset(data)

print(f"Training rows: {len(prepared.X_train):,}")
print(f"Test rows: {len(prepared.X_test):,}")

print("\nTraining class distribution")
display(class_distribution(prepared.y_train))

print("\nTest class distribution")
display(class_distribution(prepared.y_test))

## 2. Class Imbalance Review

This checks how many bookings belong to each target class. Cancelled
bookings are the minority class, but the imbalance is moderate rather than
extreme. Therefore, the baseline model is kept without class weighting so
the model can learn from the natural class distribution. The notebook still
calculates balanced class weights as a comparison to show what would happen
if Logistic Regression paid more attention to cancelled cases during
training.

In [ ]:
training_distribution = class_distribution(prepared.y_train)
negative_count = int((prepared.y_train == 0).sum())
positive_count = int((prepared.y_train == 1).sum())
total_count = len(prepared.y_train)

class_weights = {
    0: total_count / (2 * negative_count),
    1: total_count / (2 * positive_count),
}

print("Balanced class weights used by LogisticRegression(class_weight='balanced'):")
print(class_weights)
display(training_distribution)

## 3. Baseline Model

The baseline model uses standard Logistic Regression without class
weighting. Since the class imbalance is only moderate, weighting is not used
as the default setting because it can increase recall for cancellations but
also reduce precision. This gives a simple linear benchmark, showing how
well the shared predictors work before adding imbalance handling or
hyperparameter tuning.

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=RANDOM_STATE,
)
logistic_pipeline = build_pipeline(logistic_model, prepared.X_train)
logistic_pipeline

## 4. Baseline Cross-Validation

Cross-validation evaluates the baseline on several training folds before
touching the final test set. This helps estimate how stable the model is and
whether its validation performance is strong enough to compare with more
complex models.

In [ ]:
logistic_cv = cross_validation_summary(
    "Logistic Regression",
    "Baseline",
    logistic_pipeline,
    prepared.X_train,
    prepared.y_train,
)
display(logistic_cv.round(4))

**Interpretation:** Logistic Regression shows how well a simple linear model
can separate cancelled and non-cancelled bookings after the shared
preprocessing step. The cross-validation metrics are used to judge expected
generalization before looking at the final test set. A small gap between
training and validation scores indicates limited overfitting.

## 5. Baseline Final Test Evaluation

This evaluates the fitted baseline on the untouched test set. These results
show how the simple Logistic Regression model performs on unseen bookings
after model selection decisions were made using training data only.

In [ ]:
logistic_pipeline, logistic_test = fit_and_evaluate(
    "Logistic Regression",
    "Baseline",
    logistic_pipeline,
    prepared,
)

logistic_test_results = pd.DataFrame([logistic_test])
display(logistic_test_results.round(4))

**Test-set interpretation:** The final test metrics should be compared with
the cross-validation results. If the values are close, the model is behaving
consistently on unseen data. For this dataset, Logistic Regression is useful
as a stable and interpretable baseline, but its recall should be checked
carefully because low recall means more actual cancellations are missed.

## 6. Baseline Confusion Matrix

The confusion matrix shows the actual error counts behind the summary
metrics. It makes clear how many cancelled bookings were missed and how many
non-cancelled bookings were incorrectly flagged as cancellations.

In [ ]:
logistic_predictions = logistic_pipeline.predict(prepared.X_test)
logistic_confusion = confusion_matrix_frame(prepared.y_test, logistic_predictions)
display(logistic_confusion)

## 7. Class-Weighted Logistic Regression

This model adds balanced class weighting to reduce the effect of class
imbalance. The aim is to improve recall for cancelled bookings by making
mistakes on the minority class more costly during training.

In [ ]:
weighted_logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
weighted_logistic_pipeline = build_pipeline(weighted_logistic_model, prepared.X_train)
weighted_logistic_pipeline

## 8. Class-Weighted Cross-Validation

Cross-validation checks whether class weighting consistently improves the
training-fold results. This section is mainly used to compare recall and F1
against the baseline model.

In [ ]:
weighted_logistic_cv = cross_validation_summary(
    "Logistic Regression",
    "Class Weighted",
    weighted_logistic_pipeline,
    prepared.X_train,
    prepared.y_train,
)
display(weighted_logistic_cv.round(4))

## 9. Class-Weighted Final Test Evaluation

This applies the class-weighted model to the final test set. The result
shows whether the expected recall improvement also appears on unseen data,
and whether precision drops as a trade-off.

In [ ]:
weighted_logistic_pipeline, weighted_logistic_test = fit_and_evaluate(
    "Logistic Regression",
    "Class Weighted",
    weighted_logistic_pipeline,
    prepared,
)

weighted_logistic_test_results = pd.DataFrame([weighted_logistic_test])
display(weighted_logistic_test_results.round(4))

## 10. Class-Weighted Confusion Matrix

This confusion matrix explains the effect of class weighting in concrete
counts. It shows whether the model caught more actual cancellations and how
many extra false cancellation predictions were created.

In [ ]:
weighted_logistic_predictions = weighted_logistic_pipeline.predict(prepared.X_test)
weighted_logistic_confusion = confusion_matrix_frame(prepared.y_test, weighted_logistic_predictions)
display(weighted_logistic_confusion)

## 11. Baseline vs Class-Weighted Comparison

This table places the baseline and class-weighted models side by side. The
purpose is to decide whether the recall gain from class weighting is worth
the precision loss for the hotel cancellation problem.

In [ ]:
logistic_test_results = pd.DataFrame([
    logistic_test,
    weighted_logistic_test,
])
logistic_cv_results = pd.concat([
    logistic_cv,
    weighted_logistic_cv,
], ignore_index=True)

display(logistic_test_results.round(4))
display(logistic_cv_results.round(4))

**Class-imbalance interpretation:** The class-weighted model gives the
cancellation class a larger penalty during training. This usually increases
recall for cancelled bookings, but it can reduce precision because more
bookings are predicted as cancellations. Since the imbalance in this dataset
is moderate, class weighting is treated as an experiment rather than the
main model setting. The final choice should depend on whether missing
cancellations is more costly than extra follow-up actions.

## 12. Logistic Regression Hyperparameter Tuning

This grid search tunes regularization strength and class weighting with
training data only. Tuning is used to find the best model settings instead
of manually choosing them, especially the regularization level that controls
how simple or flexible the coefficient values can be. F1-score is used for
refitting because the cancellation class needs a practical balance between
precision and recall.

In [ ]:
tuned_logistic_base = build_pipeline(
    LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ),
    prepared.X_train,
)

logistic_param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

logistic_search = GridSearchCV(
    estimator=tuned_logistic_base,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=get_cv(),
    n_jobs=1,
    return_train_score=True,
)
logistic_search.fit(prepared.X_train, prepared.y_train)

print("Best Logistic Regression parameters:")
print(logistic_search.best_params_)
print(f"Best CV F1: {logistic_search.best_score_:.4f}")

**Tuning choice:** `C` controls the strength of regularization. Smaller
values create a simpler model by shrinking coefficients more strongly,
while larger values allow coefficients to fit the training data more
closely. Class weighting is included only as a checked option because the
class imbalance is moderate; the grid search can confirm whether weighting
is useful or whether the normal unweighted model gives the better F1
trade-off.

## 13. Tuned Cross-Validation

The best grid-search model is rechecked with the common cross-validation
summary. This confirms whether the selected settings give stable F1,
precision and recall compared with the earlier versions.

In [ ]:
tuned_logistic_pipeline = logistic_search.best_estimator_

tuned_logistic_cv = cross_validation_summary(
    "Logistic Regression",
    "Tuned",
    tuned_logistic_pipeline,
    prepared.X_train,
    prepared.y_train,
)
display(tuned_logistic_cv.round(4))

## 14. Tuned Final Test Evaluation

This section evaluates the tuned model on the untouched test set. It shows
whether the best cross-validation configuration still performs well when
applied to bookings that were not used during tuning.

In [ ]:
tuned_logistic_pipeline, tuned_logistic_test = fit_and_evaluate(
    "Logistic Regression",
    "Tuned",
    tuned_logistic_pipeline,
    prepared,
)

tuned_logistic_test_results = pd.DataFrame([tuned_logistic_test])
display(tuned_logistic_test_results.round(4))

## 15. Tuned Confusion Matrix

The tuned confusion matrix shows the final error pattern for Logistic
Regression. It helps explain whether tuning mainly changed missed
cancellations, false alarms, or both.

In [ ]:
tuned_logistic_predictions = tuned_logistic_pipeline.predict(prepared.X_test)
tuned_logistic_confusion = confusion_matrix_frame(prepared.y_test, tuned_logistic_predictions)
display(tuned_logistic_confusion)

## 16. Baseline vs Class-Weighted vs Tuned Comparison

This combined comparison summarizes all Logistic Regression versions. It is
used to identify the most useful version and to explain why the selected
version performs better or differently from the simple baseline.

In [ ]:
logistic_test_results = pd.DataFrame([
    logistic_test,
    weighted_logistic_test,
    tuned_logistic_test,
])
logistic_cv_results = pd.concat([
    logistic_cv,
    weighted_logistic_cv,
    tuned_logistic_cv,
], ignore_index=True)

display(logistic_test_results.round(4))
display(logistic_cv_results.round(4))

**Optimization interpretation:** The tuned model should be selected only if
it improves the chosen validation objective or gives a better practical
precision-recall trade-off on the final test set. If tuning selects
`class_weight='balanced'`, that supports the earlier imbalance experiment.
If it selects no class weighting, that means the extra recall was not worth
the precision loss under the F1 objective.

## 17. Coefficients and Odds Ratios

Logistic Regression is interpretable, so this section converts fitted
coefficients into odds ratios. These values explain which encoded predictors
increase or decrease the estimated chance of cancellation.

In [ ]:
def coefficient_table(pipeline, version, top_n=15):
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
    coefficients = pipeline.named_steps["model"].coef_[0]
    frame = pd.DataFrame({
        "version": version,
        "feature": feature_names,
        "coefficient": coefficients,
        "odds_ratio": np.exp(coefficients),
        "absolute_coefficient": np.abs(coefficients),
    })
    return frame.sort_values("absolute_coefficient", ascending=False).head(top_n)

baseline_coefficients = coefficient_table(logistic_pipeline, "Baseline")
weighted_coefficients = coefficient_table(weighted_logistic_pipeline, "Class Weighted")
tuned_coefficients = coefficient_table(tuned_logistic_pipeline, "Tuned")

print("Top baseline coefficients")
display(baseline_coefficients.round(4))

print("Top class-weighted coefficients")
display(weighted_coefficients.round(4))

print("Top tuned coefficients")
display(tuned_coefficients.round(4))

**Coefficient interpretation:** Positive coefficients increase the log-odds
of cancellation, while negative coefficients reduce them. Odds ratios above
1 indicate higher cancellation odds; values below 1 indicate lower
cancellation odds. The class-weighted coefficients show how the model shifts
when cancelled bookings receive more importance during fitting. The tuned
coefficient table explains which variables remain most influential after
the selected regularization and weighting choices.

## 18. ROC and Precision-Recall Curves

These curves compare the ranking ability of the Logistic Regression
versions across different thresholds. ROC curves show overall separation,
while precision-recall curves are especially useful for the cancelled class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_estimator(logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[0], name="Baseline")
RocCurveDisplay.from_estimator(weighted_logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[0], name="Class Weighted")
RocCurveDisplay.from_estimator(tuned_logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[0], name="Tuned")
PrecisionRecallDisplay.from_estimator(logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[1], name="Baseline")
PrecisionRecallDisplay.from_estimator(weighted_logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[1], name="Class Weighted")
PrecisionRecallDisplay.from_estimator(tuned_logistic_pipeline, prepared.X_test, prepared.y_test, ax=axes[1], name="Tuned")
axes[0].set_title("Logistic Regression ROC Curves")
axes[1].set_title("Logistic Regression Precision-Recall Curves")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "logistic_regression_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Interpretation Notes

Logistic Regression should remain in the final comparison as the interpretable
linear model. The baseline version shows normal performance at the natural
class distribution, the class-weighted version directly addresses the
imbalance between cancelled and non-cancelled bookings, and the tuned version
records the best cross-validated regularization and weighting choice. The
coefficient tables provide the explanation layer needed for the final report.